# Results Analysis — CrewAI LaTeX Article Generator

This notebook analyses the outputs produced by the 4-agent CrewAI pipeline for the topic
**"Machine Learning in Healthcare"**.

Sections:
1. ML Algorithm Performance Comparison (AUC-ROC, F1)
2. Subgroup Fairness Analysis
3. Computational Efficiency Comparison
4. API Cost Analysis by Agent
5. Pipeline Token Usage Breakdown


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11
print('Libraries loaded.')

## 1. ML Algorithm Performance Comparison

Data from the generated article (Section 3 — Results).
Three algorithms evaluated on MIMIC-III ICU mortality prediction:
- **Random Forest (RF)**
- **XGBoost (XGB)**
- **Deep Neural Network (DNN)**

Metrics: AUC-ROC under 5-fold cross-validation and temporal validation.

In [ ]:
algorithms = ['Random Forest', 'XGBoost', 'Deep Neural Network']

# AUC-ROC values from article Section 3.1
cv_auc   = [0.861, 0.893, 0.879]
temp_auc = [0.839, 0.871, 0.854]

# 95% CI half-widths (cross-validation only)
cv_ci    = [0.014, 0.012, 0.014]

x = np.arange(len(algorithms))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, cv_auc,   width, label='5-fold CV',         color='#4C72B0', yerr=cv_ci, capsize=5)
bars2 = ax.bar(x + width/2, temp_auc, width, label='Temporal Validation', color='#DD8452')

ax.set_xlabel('Algorithm')
ax.set_ylabel('AUC-ROC')
ax.set_title('AUC-ROC Comparison: Cross-Validation vs Temporal Validation')
ax.set_xticks(x)
ax.set_xticklabels(algorithms)
ax.set_ylim(0.80, 0.92)
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../results/ml_accuracy.png', dpi=150)
plt.show()
print('Figure saved to results/ml_accuracy.png')

In [ ]:
# Summary table
df_perf = pd.DataFrame({
    'Algorithm':          algorithms,
    'CV AUC-ROC':         cv_auc,
    'CV 95% CI (±)':      cv_ci,
    'Temporal AUC-ROC':   temp_auc,
    'Temporal F1':        [0.68, 0.74, 0.71],
})
df_perf = df_perf.set_index('Algorithm')
df_perf.style.highlight_max(subset=['CV AUC-ROC', 'Temporal AUC-ROC', 'Temporal F1'],
                             color='#d4edda').format(precision=3)

### Interpretation

XGBoost achieves the best performance on both validation regimes:
- **CV AUC-ROC = 0.893** — highest among all three models, with the narrowest confidence interval.
- **Temporal AUC-ROC = 0.871** — smallest degradation from CV to temporal split (Δ = 0.022 vs. 0.025 for DNN and RF).

The consistent ranking **XGBoost > DNN > RF** suggests that gradient boosting's built-in regularisation
and feature-interaction modelling are well-suited to the mixed tabular EHR feature space.

## 2. Subgroup Fairness Analysis

AUC-ROC stratified by self-reported race (temporal validation set).

In [ ]:
groups   = ['White', 'Black', 'Hispanic', 'Asian', 'Other']
xgb_auc  = [0.891,  0.861,   0.872,      0.878,   0.865]
dnn_auc  = [0.872,  0.843,   0.854,      0.860,   0.847]
rf_auc   = [0.852,  0.822,   0.832,      0.840,   0.827]

x = np.arange(len(groups))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, xgb_auc, width, label='XGBoost',            color='#4C72B0')
ax.bar(x,         dnn_auc, width, label='Deep Neural Network', color='#DD8452')
ax.bar(x + width, rf_auc,  width, label='Random Forest',       color='#55A868')

ax.set_xlabel('Patient Subgroup')
ax.set_ylabel('AUC-ROC (Temporal Validation)')
ax.set_title('Fairness Analysis: AUC-ROC by Race/Ethnicity Subgroup')
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylim(0.80, 0.91)
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# Highlight disparity for XGBoost
ax.annotate('Δ = 0.030\n(White − Black)', xy=(0.5, 0.862), xytext=(1.2, 0.858),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

plt.tight_layout()
plt.show()

In [ ]:
df_fair = pd.DataFrame({
    'Subgroup': groups,
    'XGBoost':  xgb_auc,
    'DNN':      dnn_auc,
    'RF':       rf_auc,
}).set_index('Subgroup')

# Max disparity per model
for col in df_fair.columns:
    disparity = df_fair[col].max() - df_fair[col].min()
    print(f'{col:25s} max disparity: {disparity:.3f} AUC points')

### Interpretation

All three models exhibit a performance gap between White and Black patients.
XGBoost's gap (0.030 AUC) is the largest in absolute terms, driven by the White cohort achieving
the highest AUC of any subgroup. This aligns with the known underrepresentation of minority
populations in the MIMIC-III training cohort.

**Mitigation strategies considered:**
- Adversarial de-biasing during training
- Stratified SMOTE per demographic subgroup
- Post-hoc threshold calibration per subgroup

## 3. Computational Efficiency Comparison

In [ ]:
train_times = [22, 12, 47]   # minutes, from article Section 3.4
colors = ['#55A868', '#4C72B0', '#DD8452']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Training time bar chart
axes[0].bar(algorithms, train_times, color=colors)
axes[0].set_ylabel('Training Time (minutes)')
axes[0].set_title('Training Time on Standard CPU Server')
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7)
axes[0].set_axisbelow(True)
for i, v in enumerate(train_times):
    axes[0].text(i, v + 0.5, f'{v} min', ha='center', va='bottom', fontweight='bold')

# Efficiency vs performance scatter
axes[1].scatter(train_times, temp_auc, s=150, c=colors, zorder=5)
for i, alg in enumerate(algorithms):
    axes[1].annotate(alg, (train_times[i], temp_auc[i]),
                     xytext=(5, 3), textcoords='offset points', fontsize=9)
axes[1].set_xlabel('Training Time (minutes)')
axes[1].set_ylabel('Temporal AUC-ROC')
axes[1].set_title('Performance vs Computational Cost')
axes[1].yaxis.grid(True, linestyle='--', alpha=0.7)
axes[1].xaxis.grid(True, linestyle='--', alpha=0.7)
axes[1].set_axisbelow(True)

plt.suptitle('Computational Efficiency Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Interpretation

XGBoost occupies the optimal position on the performance-vs-efficiency Pareto frontier:
- Fastest training time (12 min) while achieving the highest AUC-ROC (0.871).
- The DNN requires 4× more compute for a 0.017 AUC penalty.

This makes XGBoost the preferred choice for resource-constrained hospital environments where
GPU infrastructure is not available.

## 4. API Cost Analysis by Agent

Token usage for a single pipeline run using **Claude Sonnet 4.6**.
Pricing: $3.00 / 1M input tokens, $15.00 / 1M output tokens.

In [ ]:
INPUT_RATE  = 3.00   # USD per 1M tokens
OUTPUT_RATE = 15.00  # USD per 1M tokens

agents        = ['ResearcherAgent', 'WriterAgent', 'ReviewerAgent', 'LaTeXFormatterAgent']
input_tokens  = [1_200,  2_500,  3_200,  4_000]
output_tokens = [  800,  3_000,  1_500,  2_000]

input_costs  = [(t / 1e6) * INPUT_RATE  for t in input_tokens]
output_costs = [(t / 1e6) * OUTPUT_RATE for t in output_tokens]
total_costs  = [i + o for i, o in zip(input_costs, output_costs)]

df_cost = pd.DataFrame({
    'Agent':         agents,
    'Input Tokens':  input_tokens,
    'Output Tokens': output_tokens,
    'Input Cost $':  input_costs,
    'Output Cost $': output_costs,
    'Total Cost $':  total_costs,
}).set_index('Agent')

# Add totals row
totals = df_cost.sum()
totals.name = 'TOTAL'
df_cost = pd.concat([df_cost, totals.to_frame().T])

df_cost.style.format({
    'Input Tokens':  '{:,.0f}',
    'Output Tokens': '{:,.0f}',
    'Input Cost $':  '${:.4f}',
    'Output Cost $': '${:.4f}',
    'Total Cost $':  '${:.4f}',
}).highlight_max(subset=['Total Cost $'], color='#fff3cd')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stacked cost breakdown
x = np.arange(len(agents))
p1 = axes[0].bar(x, input_costs,  label='Input cost',  color='#4C72B0')
p2 = axes[0].bar(x, output_costs, label='Output cost', color='#DD8452',
                 bottom=input_costs)
axes[0].set_xticks(x)
axes[0].set_xticklabels([a.replace('Agent', '\nAgent') for a in agents], fontsize=9)
axes[0].set_ylabel('Cost (USD)')
axes[0].set_title('Cost Breakdown per Agent')
axes[0].legend()
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7)
axes[0].set_axisbelow(True)

# Cost share pie chart
axes[1].pie(total_costs, labels=agents, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
            startangle=140)
axes[1].set_title(f'Cost Share (Total: ${sum(total_costs):.4f})')

plt.suptitle('Claude Sonnet 4.6 API Cost Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Pipeline Token Usage Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Token volume comparison
x = np.arange(len(agents))
width = 0.35
axes[0].bar(x - width/2, input_tokens,  width, label='Input tokens',  color='#4C72B0')
axes[0].bar(x + width/2, output_tokens, width, label='Output tokens', color='#DD8452')
axes[0].set_xticks(x)
axes[0].set_xticklabels([a.replace('Agent', '\nAgent') for a in agents], fontsize=9)
axes[0].set_ylabel('Tokens')
axes[0].set_title('Token Volume per Agent')
axes[0].legend()
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7)
axes[0].set_axisbelow(True)

# Cumulative context growth
cumulative_context = np.cumsum(output_tokens)
axes[1].plot(agents, cumulative_context, marker='o', color='#4C72B0', linewidth=2)
axes[1].fill_between(range(len(agents)), cumulative_context, alpha=0.15, color='#4C72B0')
axes[1].set_xticklabels([a.replace('Agent', '\nAgent') for a in agents], fontsize=9)
axes[1].set_ylabel('Cumulative Output Tokens')
axes[1].set_title('Cumulative Context Growth Through Pipeline')
axes[1].yaxis.grid(True, linestyle='--', alpha=0.7)
axes[1].set_axisbelow(True)
for i, v in enumerate(cumulative_context):
    axes[1].annotate(f'{v:,}', (i, v), xytext=(0, 6), textcoords='offset points',
                     ha='center', fontsize=9)

plt.suptitle('Token Usage Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Total input tokens:  {sum(input_tokens):,}')
print(f'Total output tokens: {sum(output_tokens):,}')
print(f'Total tokens:        {sum(input_tokens)+sum(output_tokens):,}')
print(f'Total cost:          ${sum(total_costs):.4f}')

## Summary

| Metric | Value |
|---|---|
| Best algorithm (AUC-ROC) | XGBoost — 0.871 temporal |
| Best algorithm (F1-score) | XGBoost — 0.74 temporal |
| Fastest training | XGBoost — 12 min CPU |
| Max subgroup disparity | XGBoost — 0.030 AUC (White vs. Black) |
| Total API cost (single run) | $0.142 using Claude Sonnet 4.6 |
| Total tokens consumed | 18,200 (10,900 in / 7,300 out) |

**Recommendation:** Deploy XGBoost as the primary model in resource-constrained clinical settings,
subject to mandatory fairness auditing across demographic subgroups prior to production rollout.
Federated learning across multiple hospital sites should be explored in future work to reduce
subgroup performance gaps.